In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
from model.componentModel               import SystemModel
from model.componentModel               import SubclassModelCheckpoint

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-11-29 09:03:20.508650: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-29 09:03:20.692854: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-29 09:03:21.373191: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/ducan/anaconda3/envs/tensorflow2.10.0Crusher/lib/
2024-11-29 09:03:21.373271: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerro

In [2]:
preData                                 = PreProcessData(config = myConfig)                    # Preprocessing data
cr1, cr2, cv2                           = preData.loadH5File(name = "WITHOUTNonCriticalData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file


target_time = pd.Timestamp('2023-10-01 00:00:00')

rowCr1 = cr1.index[pd.to_datetime(cr1['tiempo']) == target_time][0]

rowCr2 = cr2.index[pd.to_datetime(cr2['tiempo']) == target_time][0]

rowCv2 = cv2.index[pd.to_datetime(cv2['tiempo']) == target_time][0]

# Display the row
print(rowCr1, rowCr2, rowCv2)

trainCr1 = cr1.iloc[:rowCr1]
trainCr2 = cr2.iloc[:rowCr2]
trainCv2 = cv2.iloc[:rowCv2]

# --------------------------------
# Add hour column into the file
# print(cv2.columns)
withHourCr1             = preData.addHours(cr1)
withHourCr2             = preData.addHours(cr2)
withHourCv2             = preData.addHours(cv2)

withIDCr1, withIDCr2, withIDCv2, IdSys  = preData.attachIDPerEvent(df = [withHourCr1,withHourCr2,withHourCv2], mainternanceAsCriticalSd = True)


# handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
handledValues           = preData.conveyorValue + preData.crusher2ConveyorValue
# -----------------------------
# Normalize
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = handledValues)  
# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data
# -----------------------------
# Extract before split
dataCv2                = withTimeDataCv2[handledValues]
labelRUL               = withTimeDataCv2["rul"]

Process:12972||2024-11-29 09:03:22,798-INFO [preProcessingData.py:loadH5File():575]:Loading H5 file ...
Process:12972||2024-11-29 09:03:23,974-INFO [preProcessingData.py:loadH5File():581]:Finish loading data from files ...
Process:12972||2024-11-29 09:03:24,030-INFO [preProcessingData.py:attachIDPerEvent():422]:Attaching ID into trajectories


225978 352528 339949


Process:12972||2024-11-29 09:03:39,037-INFO [preProcessingData.py:normalizeBuilinFunction():1439]:Normalizing the data ....


In [3]:
model       = SystemModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inFeatureDim"]))
model.summary()
model.load_weights(model_dir = "checkpoint/bestModel")


# (522340, 60, 32) (522340, 1) (522340, 60, 30) (522340, 1) (522340, 60, 2) (522340, 1) (522340, 1)

2024-11-29 09:03:39.730139: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 09:03:39.730446: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 09:03:39.736810: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 09:03:39.737068: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 09:03:39.737317: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection (SensorSel  multiple                 0 (unused)
 ection)                                                         
                                                                 
 encoder (Functional)        [(None, 1),               114806    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 31)            114431    
                                                                 
 hi_system_and_rul (HiSystem  multiple                 6413      
 AndRul)                                                         
                                                                 
Total params: 235,692
Trainable params: 235,68

Process:12972||2024-11-29 09:03:42,013-INFO [componentModel.py:load_weights():780]:Loading the model at the direction: checkpoint/bestModel
Process:12972||2024-11-29 09:03:42,047-INFO [componentModel.py:load_weights():790]:Loaded the model at the direction:checkpoint/bestModel successfully!!!


In [4]:
results            = {}
results["cv2"]     = []
# results["cr2"]     = []
# results["cv2"]     = []
# results["sys"]     = []
results["rul"]     = []
listRUL         = []


probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCv2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr2,seqRUL                      = preData.getTensorData(dataList   = [dataCv2,labelRUL], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCv2, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr2)
    # print(predictedRul)
    results["cv2"].append(zCv2)
    results["rul"].append(predictedRul.numpy())
    listRUL.append(seqRUL[:,-1,:])

Process:12972||2024-11-29 09:03:42,060-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


0


2024-11-29 09:03:43.168974: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2024-11-29 09:03:44.009514: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-11-29 09:03:44.009964: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-11-29 09:03:44.009984: W tensorflow/stream_executor/gpu/asm_compiler.cc:80] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2024-11-29 09:03:44.010420: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-11-29 09:03:44.010475: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] INTERNAL: Failed to launch ptxas
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.
Process:12972||2024-11-29 09:03:44,805-INFO [preProcessingData.

1


Process:12972||2024-11-29 09:03:46,150-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


2


... (98 outputs left out)

Process:12972||2024-11-29 09:04:18,660-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


79


Process:12972||2024-11-29 09:04:19,456-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


80


In [5]:
def calculate_mae_for_numpy(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("The two lists must have the same length.")
    
    maes = []  # To store the MAE for each pair of arrays
    for arr1, arr2 in zip(list1, list2):
        if arr1.shape != arr2.shape:
            raise ValueError("Each pair of arrays must have the same shape.")
        mae = np.mean(np.abs(arr1 - arr2))  # Element-wise absolute difference, then mean
        maes.append(mae)
    
    overall_mae = np.mean(maes)  # Average MAE across all pairs
    return overall_mae

# Example usage

print(listRUL[0].shape, results["rul"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rul"])
print("Overall MAE:", mae)



(5996, 1) (5996, 1)
Overall MAE: 42.27409821258921


In [6]:
# results            = {}
# results["cr1"]     = []
# # results["cr2"]     = []
# # results["cv2"]     = []
# # results["sys"]     = []
# results["rul"]     = []

# probsAll        = []
# binaryMaskAll   = []
# trajectID       = withIDCr1["trajectoryID"].astype(int)
# for idx in range(trajectID.max()):
#     print(idx)
#     count = (trajectID == (idx + 1)).sum() 
#     if count <= preData.config['seqLength']:
#         continue
#     seqDataCr1                              = preData.getTensorData(dataList   =    [dataCr1], 
#                                                                     trajectId   = trajectID,
#                                                                     seqLength  = preData.config['seqLength'], 
#                                                                     overlap    = preData.config['seqLength']-1,
#                                                                     getAtIdx   = idx + 1,
#                                                                     typeInput  = "listOfPandas")
    
#     zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr1[0])
#     # print(predictedRul)
#     results["cr1"].append(zCr1)
#     results["rul"].append(predictedRul)

### MASK

##### Crusher1

In [7]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr1)}")
for idx,idxSS in enumerate(handledValues):
    ssTable.add_row([idx,idxSS,probsMaskCr1[idx].numpy(),binaryMaskCr1[idx].numpy()])

print(ssTable)

TypeError: 'NoneType' object is not iterable

##### Crusher 2

In [ ]:
# ssTable                 = PrettyTable()
# ssTable.field_names     = ["No","sensorName","probability", "binary"]
# print(f"The number of selected sensor {sum(binaryMaskCr2)}")
# for idx,idxSS in enumerate(preData.crusher2Value):
#     ssTable.add_row([idx,idxSS,probsMaskCr2[idx].numpy(),binaryMaskCr2[idx].numpy()])

# print(ssTable)

##### ConveyorBelt 2

In [ ]:
# ssTable                 = PrettyTable()
# ssTable.field_names     = ["No","sensorName","probability", "binary"]
# print(f"The number of selected sensor {sum(binaryMaskCv2)}")
# for idx,idxSS in enumerate(preData.conveyorValue):
#     ssTable.add_row([idx,idxSS,probsMaskCv2[idx].numpy(),binaryMaskCv2[idx].numpy()])

# print(ssTable)

### Health indicators

##### Crusher 1

In [ ]:
print(len(results))

2


In [8]:
fig = {}
for idxResults in results.keys():
    fig[idxResults]     = go.Figure()
    current_time        = 0
    
    for idx, signal in enumerate(results[idxResults]):
        # print(signal)
        signal          = tf.squeeze(signal).numpy()
        # print(idx)
        # print(signal)
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[idxResults].add_trace(go.Scatter(
                                                x   = time_vector,
                                                y   = signal,
                                                mode= 'lines',
                                                name= f'Signal {idx + 1}'  # Label each signal for legend toggling
        ))
    # Customize layout for readability
    fig[idxResults].update_layout(
                                    title       = idxResults,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[idxResults].show()


##### Crusher 2

In [ ]:
# fig1             = go.Figure()
# current_time    = 0

# for idx, signal in enumerate(results["cr2"]):
#     signal          = tf.squeeze(signal).numpy()
#     time_vector     = np.arange(current_time, current_time + len(signal))
#     current_time    += len(signal)
    
#     # Add each signal as a separate trace
#     fig1.add_trace(go.Scatter(
#         x=time_vector,
#         y=signal,
#         mode='lines',
#         name=f'Signal {idx + 1}'  # Label each signal for legend toggling
#     ))

# # Customize layout for readability
# fig1.update_layout(
#     title="Continuous Time Series Plot with Toggle",
#     xaxis_title="Time (arbitrary units)",
#     yaxis_title="Signal Value",
#     template="plotly_white",
#     showlegend=True  # Show legend to allow toggling each signal
# )

# # Display the plot
# fig1.show()
